# Entity Identification Pipeline for Co-Pilot Agent

## Overview
This notebook compares the **SAME MODEL** with and without fine-tuning:

1. **Pre-trained (No Fine-tuning)**: Use DistilBERT embeddings + similarity matching
2. **Fine-tuned**: Same DistilBERT model trained on our data

**Same base model: `distilbert-base-uncased`**

**No data leakage**: Test set is never seen during training.

## Entity Types
- CDR (Call Detail Records)
- Phone
- Web Activity
- Web Actor
- Person
- Investigation
- Insight
- Report
- EVisa Request

## 1. Setup and Dependencies

In [ ]:
# Install required packages
# !pip install pandas numpy scikit-learn torch transformers accelerate

import pandas as pd
import numpy as np
import json
import ast
from typing import List, Dict, Tuple
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss, jaccard_score
)

# Deep learning imports
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Define the base model - SAME for both approaches
BASE_MODEL = "distilbert-base-uncased"
print(f"\nBase model for both approaches: {BASE_MODEL}")

## 2. Load and Preprocess Data

In [ ]:
# Load datasets
user_queries_df = pd.read_csv('user_queries.csv')
fields_description_df = pd.read_csv('fields_description.csv')

print(f"User Queries: {len(user_queries_df)} rows")
print(f"Fields Description: {len(fields_description_df)} rows")

In [ ]:
def safe_parse_json(json_str: str) -> dict:
    """Parse JSON string, handling Python dict format."""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(json_str)
        except (ValueError, SyntaxError):
            return {}

def extract_entities(json_obj: dict) -> List[str]:
    """Extract entities from entityType and relationTargetType keys."""
    entities = set()
    
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    def search_statements(statements):
        if not statements:
            return
        for stmt in statements:
            if isinstance(stmt, dict):
                params = stmt.get('parameters', {})
                if 'relationTargetType' in params:
                    targets = params['relationTargetType']
                    if isinstance(targets, list):
                        entities.update(targets)
                    else:
                        entities.add(targets)
                if 'statements' in stmt:
                    search_statements(stmt['statements'])
    
    if 'statements' in json_obj:
        search_statements(json_obj['statements'])
    
    return sorted(list(entities))

# Parse and extract entities
user_queries_df['parsed_json'] = user_queries_df['json'].apply(safe_parse_json)
user_queries_df['entities'] = user_queries_df['parsed_json'].apply(extract_entities)

# Show distribution
all_entities_flat = [e for ents in user_queries_df['entities'] for e in ents]
entity_counts = Counter(all_entities_flat)
print("Entity Distribution:")
for entity, count in entity_counts.most_common():
    print(f"  {entity}: {count} ({100*count/len(user_queries_df):.1f}%)")

multi_entity = sum(1 for e in user_queries_df['entities'] if len(e) > 1)
print(f"\nQueries with multiple entities: {multi_entity} ({100*multi_entity/len(user_queries_df):.1f}%)")

In [ ]:
# Define entity labels with descriptions
ALL_ENTITIES = sorted(list(set(all_entities_flat)))
print(f"All entity types ({len(ALL_ENTITIES)}): {ALL_ENTITIES}")

# Entity descriptions for embedding-based classification
ENTITY_DESCRIPTIONS = {
    'CDR': 'Call Detail Records including phone calls, SMS text messages, emails, voice communications, and web communications between devices',
    'EVisa Request': 'Electronic visa applications, travel document requests, visitor arrivals and departures, immigration records',
    'Insight': 'Intelligence insights, analysis notes, findings, assessments, and investigative observations',
    'Investigation': 'Investigation cases, inquiries, probes, open and closed cases with priority levels',
    'Person': 'Individual people with personal information like first name, last name, birth date, gender, passport, occupation',
    'Phone': 'Phone devices and identifiers including IMEI, IMSI, MSISDN phone numbers, mobile devices, landlines, suspicious phones, target phones',
    'Report': 'Reports, documents, documentation, summaries, and briefings',
    'Web Activity': 'Social media posts, comments, tweets, online content, likes, shares, hashtags from platforms like Facebook, Twitter, Instagram, Reddit, YouTube',
    'Web Actor': 'Social media profiles, accounts, channels, pages on platforms like Facebook, Twitter, Instagram, with followers, friends, and profile information'
}

## 3. Train/Test Split (No Data Leakage)

In [ ]:
# Create label encoder
mlb = MultiLabelBinarizer(classes=ALL_ENTITIES)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

# Split: 80% train, 20% test - STRICT SEPARATION
X = user_queries_df['question'].tolist()
y = y_encoded
entities_list = user_queries_df['entities'].tolist()

indices = list(range(len(X)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

# Training data - ONLY used for fine-tuning (Approach 2)
X_train = [X[i] for i in train_idx]
y_train = y[train_idx]
train_entities = [entities_list[i] for i in train_idx]

# Test data - used for evaluation of BOTH approaches
X_test = [X[i] for i in test_idx]
y_test = y[test_idx]
test_entities = [entities_list[i] for i in test_idx]

print(f"Train size: {len(X_train)} (used ONLY for fine-tuning)")
print(f"Test size: {len(X_test)} (used for evaluation of both approaches)")
print(f"\n*** No data leakage: Test data is never seen during training ***")

## 4. Evaluation Framework

In [ ]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, 
                        label_names: List[str], method_name: str = "Method") -> Dict:
    """Compute and display evaluation metrics."""
    metrics = {
        'exact_match': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'jaccard_micro': jaccard_score(y_true, y_pred, average='micro', zero_division=0),
    }
    
    print(f"\n{'='*60}")
    print(f"RESULTS: {method_name}")
    print(f"{'='*60}")
    print(f"Exact Match Accuracy: {metrics['exact_match']:.4f}")
    print(f"F1 Micro:             {metrics['f1_micro']:.4f}")
    print(f"F1 Macro:             {metrics['f1_macro']:.4f}")
    print(f"Precision Micro:      {metrics['precision_micro']:.4f}")
    print(f"Recall Micro:         {metrics['recall_micro']:.4f}")
    print(f"Jaccard Micro:        {metrics['jaccard_micro']:.4f}")
    print(f"Hamming Loss:         {metrics['hamming_loss']:.4f}")
    print(f"\nPer-Class Report:")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    
    return metrics

---
# APPROACH 1: Pre-trained Model (No Fine-tuning)

Use the **same DistilBERT model** without any fine-tuning.
Classification via embedding similarity to entity descriptions.

In [ ]:
class PretrainedEmbeddingClassifier:
    """
    Pre-trained classifier using embedding similarity.
    NO fine-tuning - uses the model as-is.
    
    Classification approach:
    1. Encode query using pre-trained model
    2. Encode entity descriptions using same model
    3. Predict based on cosine similarity
    """
    
    def __init__(self, model_name: str = BASE_MODEL):
        self.model_name = model_name
        self.device = DEVICE
        
        print(f"Loading pre-trained model: {model_name}")
        print("*** NO fine-tuning - using model as-is ***")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        # Freeze all parameters (no training)
        for param in self.model.parameters():
            param.requires_grad = False
        
        print(f"Model loaded! Parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        
        # Pre-compute entity description embeddings
        self.entity_labels = ALL_ENTITIES
        self.entity_descriptions = ENTITY_DESCRIPTIONS
        self.entity_embeddings = None
        self._compute_entity_embeddings()
    
    def _get_embedding(self, text: str) -> np.ndarray:
        """Get embedding for a single text using mean pooling."""
        inputs = self.tokenizer(
            text, return_tensors='pt', truncation=True, 
            padding=True, max_length=128
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Mean pooling over token embeddings
            attention_mask = inputs['attention_mask']
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        
        return embedding.cpu().numpy()[0]
    
    def _compute_entity_embeddings(self):
        """Pre-compute embeddings for all entity descriptions."""
        print("Computing entity description embeddings...")
        embeddings = []
        for entity in self.entity_labels:
            desc = self.entity_descriptions[entity]
            emb = self._get_embedding(desc)
            embeddings.append(emb)
        self.entity_embeddings = np.array(embeddings)
        print(f"Entity embeddings shape: {self.entity_embeddings.shape}")
    
    def predict_single(self, query: str, threshold: float = 0.5) -> Tuple[List[str], np.ndarray]:
        """
        Predict entities for a single query using embedding similarity.
        """
        # Get query embedding
        query_embedding = self._get_embedding(query)
        
        # Compute cosine similarities
        query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-9)
        entity_norms = self.entity_embeddings / (np.linalg.norm(self.entity_embeddings, axis=1, keepdims=True) + 1e-9)
        similarities = np.dot(entity_norms, query_norm)
        
        # Convert to probabilities using softmax-like scaling
        # Scale similarities to be more discriminative
        scaled_sims = (similarities - similarities.min()) / (similarities.max() - similarities.min() + 1e-9)
        
        # Predict entities above threshold
        predictions = [self.entity_labels[i] for i, s in enumerate(scaled_sims) if s >= threshold]
        
        # If no predictions, take the top one
        if not predictions:
            predictions = [self.entity_labels[np.argmax(similarities)]]
        
        return predictions, similarities
    
    def predict_batch(self, queries: List[str], threshold: float = 0.5,
                      verbose: bool = True) -> List[List[str]]:
        """Predict entities for multiple queries."""
        predictions = []
        for i, query in enumerate(queries):
            if verbose and i % 30 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            pred, _ = self.predict_single(query, threshold)
            predictions.append(pred)
        return predictions

In [ ]:
# Initialize pre-trained classifier (NO fine-tuning)
print("="*60)
print(f"APPROACH 1: Pre-trained {BASE_MODEL} (No Fine-tuning)")
print("="*60)
print("\nThis model has NEVER been trained on our data.")
print("It uses pre-trained embeddings + similarity matching.\n")

pretrained_classifier = PretrainedEmbeddingClassifier(model_name=BASE_MODEL)

In [ ]:
# Test pre-trained classifier on a few examples
print("\nTesting pre-trained classifier:")
test_queries_sample = [
    "What SMS messages were sent from suspicious phones?",
    "Find all individuals with occupation engineer",
    "Show me tweets from accounts mentioning Tesla"
]

for query in test_queries_sample:
    pred, sims = pretrained_classifier.predict_single(query, threshold=0.5)
    print(f"\nQuery: {query}")
    print(f"Predicted: {pred}")
    top_3 = sorted(zip(ALL_ENTITIES, sims), key=lambda x: -x[1])[:3]
    print(f"Top similarities: {[(e, f'{s:.3f}') for e, s in top_3]}")

In [ ]:
# Run pre-trained predictions on TEST SET
print("\nRunning pre-trained predictions on test set...")
print("(No training was performed - using pre-trained weights only)\n")

predictions_pretrained = pretrained_classifier.predict_batch(X_test, threshold=0.5)
y_pred_pretrained = mlb.transform(predictions_pretrained)

metrics_pretrained = evaluate_predictions(
    y_test, y_pred_pretrained, ALL_ENTITIES,
    f"Approach 1: Pre-trained {BASE_MODEL} (No Fine-tuning)"
)

---
# APPROACH 2: Fine-tuned Model

Use the **same DistilBERT model** but fine-tune it on our training data.

In [ ]:
class EntityDataset(Dataset):
    """PyTorch Dataset for entity classification."""
    
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }


class FineTunedClassifier:
    """
    Fine-tuned classifier - SAME base model but trained on our data.
    """
    
    def __init__(self, model_name: str = BASE_MODEL, num_labels: int = 9):
        self.model_name = model_name
        self.num_labels = num_labels
        self.device = DEVICE
        self.model = None
        self.tokenizer = None
        self.label_names = ALL_ENTITIES
    
    def train(self, X_train: List[str], y_train: np.ndarray,
              epochs: int = 10, batch_size: int = 16, max_length: int = 128):
        """
        Train the classifier on training data ONLY.
        Test data is never used here.
        """
        print(f"\n{'='*60}")
        print(f"TRAINING: Fine-tuned {self.model_name}")
        print(f"{'='*60}")
        print(f"Base Model: {self.model_name}")
        print(f"Training samples: {len(X_train)}")
        print(f"Epochs: {epochs}")
        print("\n*** Test data is NOT used during training ***")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="multi_label_classification"
        )
        
        print(f"Model parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        
        # Split TRAINING data for train/validation (NOT using test data)
        X_t, X_v, y_t, y_v = train_test_split(
            X_train, y_train, test_size=0.15, random_state=42
        )
        
        print(f"Training split: {len(X_t)} train, {len(X_v)} validation")
        
        train_dataset = EntityDataset(X_t, y_t, self.tokenizer, max_length)
        val_dataset = EntityDataset(X_v, y_v, self.tokenizer, max_length)
        
        training_args = TrainingArguments(
            output_dir='./classifier_output',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=100,
            weight_decay=0.01,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
        )
        
        print("\nTraining...")
        trainer.train()
        print("Training complete!")
        
        self.model.eval()
        self.model.to(self.device)
    
    def predict_single(self, query: str, threshold: float = 0.5) -> Tuple[List[str], np.ndarray]:
        """Predict entities for a single query."""
        inputs = self.tokenizer(
            query,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
        
        predicted = [self.label_names[i] for i, p in enumerate(probs) if p > threshold]
        return predicted if predicted else [self.label_names[np.argmax(probs)]], probs
    
    def predict_batch(self, queries: List[str], threshold: float = 0.5,
                      verbose: bool = True) -> List[List[str]]:
        """Predict entities for multiple queries."""
        predictions = []
        for i, query in enumerate(queries):
            if verbose and i % 50 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            pred, _ = self.predict_single(query, threshold)
            predictions.append(pred)
        return predictions
    
    def save(self, path: str = './entity_classifier_model'):
        """Save the model."""
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")
    
    def load(self, path: str = './entity_classifier_model'):
        """Load a saved model."""
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path)
        self.model.eval()
        self.model.to(self.device)
        print(f"Model loaded from {path}")

In [ ]:
# Initialize and train the fine-tuned classifier
print("="*60)
print(f"APPROACH 2: Fine-tuned {BASE_MODEL}")
print("="*60)

finetuned_classifier = FineTunedClassifier(model_name=BASE_MODEL, num_labels=len(ALL_ENTITIES))

# Train ONLY on training data (X_train, y_train)
# Test data (X_test, y_test) is NEVER seen during training
finetuned_classifier.train(X_train, y_train, epochs=10, batch_size=16)

In [ ]:
# Save the fine-tuned model
finetuned_classifier.save('./entity_classifier_model')

In [ ]:
# Run fine-tuned predictions on TEST SET (never seen during training)
print("\nRunning fine-tuned predictions on test set...")
print("(Test data was NEVER seen during training)\n")

predictions_finetuned = finetuned_classifier.predict_batch(X_test, threshold=0.5)
y_pred_finetuned = mlb.transform(predictions_finetuned)

metrics_finetuned = evaluate_predictions(
    y_test, y_pred_finetuned, ALL_ENTITIES,
    f"Approach 2: Fine-tuned {BASE_MODEL}"
)

---
## 5. Results Comparison

In [ ]:
print("\n" + "="*70)
print(f"COMPARISON: Same Model ({BASE_MODEL})")
print(f"Pre-trained (No Training) vs Fine-tuned (Trained on our data)")
print("="*70)

comparison_df = pd.DataFrame({
    'Metric': ['Exact Match', 'F1 Micro', 'F1 Macro', 'Precision', 'Recall', 'Jaccard', 'Hamming Loss'],
    'Pre-trained (No Training)': [
        metrics_pretrained['exact_match'],
        metrics_pretrained['f1_micro'],
        metrics_pretrained['f1_macro'],
        metrics_pretrained['precision_micro'],
        metrics_pretrained['recall_micro'],
        metrics_pretrained['jaccard_micro'],
        metrics_pretrained['hamming_loss']
    ],
    'Fine-tuned': [
        metrics_finetuned['exact_match'],
        metrics_finetuned['f1_micro'],
        metrics_finetuned['f1_macro'],
        metrics_finetuned['precision_micro'],
        metrics_finetuned['recall_micro'],
        metrics_finetuned['jaccard_micro'],
        metrics_finetuned['hamming_loss']
    ]
})

# Calculate improvement
comparison_df['Improvement'] = comparison_df['Fine-tuned'] - comparison_df['Pre-trained (No Training)']
comparison_df['% Change'] = comparison_df.apply(
    lambda row: f"{100 * row['Improvement'] / max(abs(row['Pre-trained (No Training)']), 0.001):+.1f}%" 
    if row['Metric'] != 'Hamming Loss' 
    else f"{-100 * row['Improvement'] / max(abs(row['Pre-trained (No Training)']), 0.001):+.1f}%",
    axis=1
)

print(comparison_df.to_string(index=False))

print("\n" + "-"*70)
print("SUMMARY:")
print(f"  Base Model: {BASE_MODEL}")
print(f"  Training samples used for fine-tuning: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")

f1_improvement = metrics_finetuned['f1_micro'] - metrics_pretrained['f1_micro']
exact_improvement = metrics_finetuned['exact_match'] - metrics_pretrained['exact_match']
print(f"\n  F1 Micro improvement:      {f1_improvement:+.4f}")
print(f"  Exact Match improvement:   {exact_improvement:+.4f}")

---
## 6. Test Cases Comparison

In [ ]:
# Define test cases
test_cases = [
    ("What SMS messages were sent from suspicious phones to 0549876543 containing 'urgent'?", ["CDR", "Phone"]),
    ("Find all calls made using 3G technology", ["CDR"]),
    ("Show me all tweets from accounts with 500 friends mentioning Tesla", ["Web Activity", "Web Actor"]),
    ("Which phones have been marked as suspicious?", ["Phone"]),
    ("Find all individuals with occupation 'engineer' born before July 1985", ["Person"]),
    ("Show me investigations that are open or created in the last 3 months", ["Investigation"]),
    ("Find insights containing 'money laundering' from the past month", ["Insight"]),
    ("List visitors whose travel document was issued before January 2020", ["EVisa Request"]),
    ("Get reports created in the past 3 days", ["Report"]),
    ("Find Instagram profiles with 100 followers using Israel phone number", ["Web Actor"]),
    ("List emails sent to phones associated with target Sarah Johnson", ["CDR", "Phone"]),
]

print("\n" + "="*70)
print("TEST CASES COMPARISON")
print(f"Same model: {BASE_MODEL}")
print("="*70)

def check_prediction(pred, expected):
    """Check if prediction matches expected."""
    pred_set, exp_set = set(pred), set(expected)
    if pred_set == exp_set:
        return "✓ EXACT"
    elif pred_set & exp_set:
        return "~ PARTIAL"
    else:
        return "✗ WRONG"

pt_correct = 0
ft_correct = 0

for query, expected in test_cases:
    # Pre-trained prediction
    pt_pred, _ = pretrained_classifier.predict_single(query, threshold=0.5)
    
    # Fine-tuned prediction
    ft_pred, _ = finetuned_classifier.predict_single(query, threshold=0.5)
    
    pt_result = check_prediction(pt_pred, expected)
    ft_result = check_prediction(ft_pred, expected)
    
    if "EXACT" in pt_result:
        pt_correct += 1
    if "EXACT" in ft_result:
        ft_correct += 1
    
    print(f"\nQuery: {query[:65]}{'...' if len(query) > 65 else ''}")
    print(f"Expected: {expected}")
    print(f"  Pre-trained: {pt_result:12} {pt_pred}")
    print(f"  Fine-tuned:  {ft_result:12} {ft_pred}")

print(f"\n" + "="*70)
print(f"Test Cases Summary:")
print(f"  Pre-trained: {pt_correct}/{len(test_cases)} exact matches ({100*pt_correct/len(test_cases):.0f}%)")
print(f"  Fine-tuned:  {ft_correct}/{len(test_cases)} exact matches ({100*ft_correct/len(test_cases):.0f}%)")

---
## 7. Summary and Conclusions

### Same Model Comparison

| Aspect | Pre-trained (No Fine-tuning) | Fine-tuned |
|--------|------------------------------|------------|
| **Base Model** | distilbert-base-uncased | distilbert-base-uncased |
| **Training Data** | None | Training set (80%) |
| **Test Data Seen** | Never | Never |
| **Classification Method** | Embedding similarity | Learned classifier head |
| **Inference Speed** | Similar | Similar |

### Key Findings

1. **Same Architecture**: Both approaches use the exact same DistilBERT model
2. **Fair Comparison**: Only difference is whether the model was fine-tuned
3. **No Data Leakage**: Test set was never seen during training

### Why Fine-tuning Helps
- Pre-trained model has general language understanding but doesn't know our specific entity definitions
- Fine-tuning teaches the model the patterns specific to this domain and dataset
- The classification head learns to map from embeddings to our specific entity labels

### Metrics Explanation
- **Exact Match**: % of queries where ALL predicted entities exactly match ground truth
- **F1 Score**: Harmonic mean of precision and recall
- **Hamming Loss**: Fraction of incorrectly predicted labels (lower is better)

### Open Issues & Future Improvements
1. **Better Entity Descriptions**: Improve descriptions for better pre-trained performance
2. **Few-Shot Learning**: Use a small number of examples without full fine-tuning
3. **Larger Models**: Try BERT-base or RoBERTa for potentially better results
4. **Ensemble**: Combine both approaches

In [ ]:
# Save comparison results
comparison_df.to_csv('approach_comparison_results.csv', index=False)
print("Results saved to approach_comparison_results.csv")